# 🚀 Модуль 7. Урок 4. P-hacking, Peeking Problem и множественное тестирование

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_07_ab_testing/lesson_04_p_hacking_peeking_problem.ipynb)

## 📌 Бизнес-контекст и методология
Peeking Problem (подглядывание) возникает, когда аналитик проверяет p-value каждый день и останавливает эксперимент при первом получении $p < 0.05$.
Это завышает вероятность ложноположительного вывода (False Positive) с 5% до 30-40%!

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import numpy as np
from scipy import stats

np.random.seed(42)
n_experiments = 300
false_positives = 0

for _ in range(n_experiments):
    a = np.random.normal(0, 1, 1000)
    b = np.random.normal(0, 1, 1000)
    for check_point in [200, 400, 600, 800, 1000]:
        _, p = stats.ttest_ind(a[:check_point], b[:check_point])
        if p < 0.05:
            false_positives += 1
            break

actual_alpha = false_positives / n_experiments
print(f"Номинальный alpha: 5.0%")
print(f"Фактический alpha при 5 подглядываниях: {actual_alpha:.1%}")


Номинальный alpha: 5.0%
Фактический alpha при 5 подглядываниях: 15.0%


### 🎯 Практическое задание 1: Поправка Бонферрони
При проверке 10 независимых метрик скорректируйте пороговый уровень значимости alpha=0.05 методом Бонферрони.

In [3]:
### ВАШ КОД ЗДЕСЬ
n_metrics = 10
alpha_bonferroni = 0.05 / n_metrics
print(f"Скорректированный альфа: {alpha_bonferroni:.4f}")

assert alpha_bonferroni == 0.005, "Поправка Бонферрони делит альфа на число тестов"
print("✅ Тест успешно пройден: защита от множественной проверки подтверждена!")


Скорректированный альфа: 0.0050
✅ Тест успешно пройден: защита от множественной проверки подтверждена!


### 💼 Как жить без подглядывания?
1. Заранее фиксировать дату окончания теста.
2. Применять последовательный анализ Вальда (mSPRT / Sequential Testing).